# 02 Data Preprocessing – COMPAS Recidivism Dataset

## 1. Purpose

This notebook prepares the COMPAS dataset for baseline modelling and the downstream Explainability–Ethics Scoring (EES) framework.

It is aligned with the German Credit and Pima Diabetes preprocessing notebooks in terms of:

- folder structure;
- output names;
- train/test split design;
- original and preprocessed feature exports;
- fitted preprocessing pipeline export;
- feature treatment documentation;
- reproducibility controls.

This version incorporates the updated COMPAS EDA decisions:

- keep only the agreed final modelling features;
- remove identifiers;
- remove COMPAS-generated scores;
- remove future-event leakage variables;
- remove duplicate variables;
- remove constant/non-informative variables;
- remove high-cardinality charge descriptions;
- retain sensitive variables for fairness/proxy-risk auditing.

## 2. Pipeline Alignment

This notebook produces the standard outputs required by later notebooks:

| Output | Purpose |
|---|---|
| `X_train_original.csv` | Original train features for full-pipeline modelling and DiCE |
| `X_test_original.csv` | Original test features for full-pipeline modelling and DiCE |
| `X_train_preprocessed.csv` | Preprocessed train features for estimator-only models, SHAP and LIME |
| `X_test_preprocessed.csv` | Preprocessed test features for estimator-only models, SHAP and LIME |
| `y_train.csv` | Training target |
| `y_test.csv` | Test target |
| `preprocessor.pkl` | Fitted preprocessing object |
| `preprocessing_pipeline.joblib` | Compatibility copy of the fitted preprocessor |
| `preprocessing_feature_registry.csv` | Feature treatment audit |
| `preprocessing_summary.csv` | High-level preprocessing summary |

These outputs are intentionally aligned with German Credit and Pima Diabetes.

## 3. COMPAS-Specific Assumptions

### Target

The target is:

```text
is_recid
```

Target interpretation:

```text
0 = no recidivism
1 = recidivism
```

Rows with invalid target values such as `-1` are excluded.

### Final retained modelling features

The final modelling feature set is:

```text
sex
age
race

juv_fel_count
juv_misd_count
juv_other_count
priors_count

days_b_screening_arrest
c_days_from_compas

c_charge_degree
```

### Why these features are retained

These features represent information available at or before assessment time:

- demographic attributes retained for fairness/proxy-risk analysis;
- criminal history information;
- assessment timing information;
- current charge severity.

### Excluded feature groups

The following feature groups are excluded:

| Group | Examples | Reason |
|---|---|---|
| Identifiers | `id`, `name`, `first`, `last`, `dob`, case numbers | No modelling meaning and privacy risk |
| COMPAS score outputs | `decile_score`, `score_text`, `v_decile_score`, `v_score_text`, `decile_score.1` | Circular prediction / leakage |
| Future-event variables | `r_charge_degree`, `r_charge_desc`, `r_days_from_arrest`, `vr_charge_degree`, `vr_charge_desc` | Known only after recidivism event |
| Duplicate variables | `priors_count.1` | Duplicates `priors_count` |
| Derived redundant variables | `age_cat` | Derived from `age` |
| High-cardinality descriptors | `c_charge_desc` | Sparse one-hot features and unstable explanations |
| Constant/non-informative fields | `type_of_assessment`, `v_type_of_assessment` | Single unique value |
| Empty variables | `violent_recid` | 100% missing |

### Preprocessing approach

- Numeric variables: median imputation + RobustScaler
- Categorical variables: missing category imputation + One-Hot Encoding
- Train/test split: 80/20 stratified split
- Random seed: 42

## 4. Defensible Reasoning

This preprocessing strategy is defensible because it:

- avoids target leakage;
- avoids circular use of COMPAS-generated scores;
- excludes identifiers and privacy-sensitive name fields;
- keeps the modelling dataset small and interpretable;
- avoids unstable high-cardinality one-hot expansions;
- retains sensitive variables for fairness, proxy-risk and explanation auditing;
- preserves both original and preprocessed feature spaces for downstream modelling, explainability and recourse analysis.

In [1]:
# =========================================================
# 5. Import Required Libraries
# =========================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler

pd.set_option("display.max_columns", 250)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

RANDOM_STATE = 42

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# =========================================================
# 6. Mount Google Drive and Configure Paths
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

ROOT_PATH = Path("/content/drive/MyDrive/Master_Research")

DATASET_NAME = "compas"
NOTEBOOK_NAME = "02_data_preprocessing_compas"

RESULTS_ROOT = ROOT_PATH / "07_Results" / DATASET_NAME

EDA_RESULTS_PATH = RESULTS_ROOT / "01_manual_eda_compas"
EDA_DATASETS_PATH = EDA_RESULTS_PATH / "datasets"
EDA_TABLES_PATH = EDA_RESULTS_PATH / "tables"

RESULTS_PATH = RESULTS_ROOT / NOTEBOOK_NAME
DATASETS_PATH = RESULTS_PATH / "datasets"
TABLES_PATH = RESULTS_PATH / "tables"
MODELS_PATH = RESULTS_PATH / "models"
LOGS_PATH = RESULTS_PATH / "logs"

for path in [
    RESULTS_PATH,
    DATASETS_PATH,
    TABLES_PATH,
    MODELS_PATH,
    LOGS_PATH
]:
    path.mkdir(parents=True, exist_ok=True)

print("Project paths configured.")
print(f"EDA datasets path: {EDA_DATASETS_PATH}")
print(f"EDA tables path:   {EDA_TABLES_PATH}")
print(f"Results path:      {RESULTS_PATH}")
print(f"Datasets path:     {DATASETS_PATH}")
print(f"Tables path:       {TABLES_PATH}")
print(f"Models path:       {MODELS_PATH}")

Mounted at /content/drive
Project paths configured.
EDA datasets path: /content/drive/MyDrive/Master_Research/07_Results/compas/01_manual_eda_compas/datasets
EDA tables path:   /content/drive/MyDrive/Master_Research/07_Results/compas/01_manual_eda_compas/tables
Results path:      /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas
Datasets path:     /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/datasets
Tables path:       /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/tables
Models path:       /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/models


In [3]:
# =========================================================
# 7. Helper Functions
# =========================================================

def read_csv_safely(path, label, required=True):
    path = Path(path)

    if not path.exists():
        if required:
            raise FileNotFoundError(f"{label} not found at: {path}")

        print(f"Optional file not found: {label} at {path}")
        return pd.DataFrame()

    return pd.read_csv(path)


def read_excel_safely(path, label, required=True):
    path = Path(path)

    if not path.exists():
        if required:
            raise FileNotFoundError(f"{label} not found at: {path}")

        print(f"Optional file not found: {label} at {path}")
        return pd.DataFrame()

    return pd.read_excel(path)


def find_first_existing_file(candidate_paths, label, required=True):
    for path in candidate_paths:
        path = Path(path)

        if path.exists():
            print(f"{label} found at: {path}")
            return path

    if required:
        checked = "\\n".join(str(path) for path in candidate_paths)
        raise FileNotFoundError(f"{label} not found. Checked:\\n{checked}")

    print(f"{label} not found.")
    return None


def save_df(df, filename, folder=None, index=False):
    if folder is None:
        folder = TABLES_PATH

    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)

    output_path = folder / filename
    df.to_csv(output_path, index=index)

    print(f"Saved: {output_path}")
    return output_path


def save_excel(dfs, filename, folder=None):
    if folder is None:
        folder = TABLES_PATH

    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)

    output_path = folder / filename

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet_name, df in dfs.items():
            if isinstance(df, pd.DataFrame):
                df.to_excel(writer, sheet_name=sheet_name[:31], index=False)

    print(f"Saved: {output_path}")
    return output_path


def get_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def get_feature_names_from_preprocessor(preprocessor, numeric_features, categorical_features):
    feature_names = []

    if numeric_features:
        feature_names.extend(numeric_features)

    if categorical_features:
        ohe = preprocessor.named_transformers_["cat"].named_steps["encoder"]
        cat_names = ohe.get_feature_names_out(categorical_features).tolist()
        feature_names.extend(cat_names)

    return feature_names

## 8. Load EDA Outputs

This notebook uses the outputs created by the updated COMPAS EDA notebook:

- `compas_selected_raw_dataset_snapshot.csv`
- `feature_inclusion_exclusion_register.csv`
- `recommended_modelling_features.csv`
- `feature_audit_table.csv`

If the EDA outputs are unavailable, the notebook falls back to the manually agreed final feature list.

In [4]:
# =========================================================
# 8. Load EDA Outputs
# =========================================================

selected_raw_path = find_first_existing_file(
    [
        EDA_DATASETS_PATH / "compas_selected_raw_dataset_snapshot.csv",
        EDA_TABLES_PATH / "compas_selected_raw_dataset_snapshot.csv",
    ],
    "Selected raw COMPAS snapshot"
)

feature_decision_path = find_first_existing_file(
    [
        EDA_TABLES_PATH / "feature_inclusion_exclusion_register.csv",
        EDA_TABLES_PATH / "feature_decision_register.csv",
    ],
    "Feature inclusion/exclusion register",
    required=False
)

recommended_features_path = find_first_existing_file(
    [
        EDA_TABLES_PATH / "recommended_modelling_features.csv"
    ],
    "Recommended modelling features",
    required=False
)

feature_audit_path = find_first_existing_file(
    [
        EDA_TABLES_PATH / "feature_audit_table.csv",
        EDA_TABLES_PATH / "feature_audit_table.xlsx"
    ],
    "Feature audit table",
    required=False
)

df = read_csv_safely(
    selected_raw_path,
    "COMPAS selected raw dataset snapshot"
)

feature_decision_df = (
    read_csv_safely(feature_decision_path, "feature inclusion/exclusion register", required=False)
    if feature_decision_path is not None
    else pd.DataFrame()
)

recommended_features_df = (
    read_csv_safely(recommended_features_path, "recommended modelling features", required=False)
    if recommended_features_path is not None
    else pd.DataFrame()
)

if feature_audit_path is not None and Path(feature_audit_path).suffix.lower() == ".csv":
    feature_audit_df = read_csv_safely(feature_audit_path, "feature audit table", required=False)
elif feature_audit_path is not None:
    feature_audit_df = read_excel_safely(feature_audit_path, "feature audit table", required=False)
else:
    feature_audit_df = pd.DataFrame()

print("Raw selected dataset shape:", df.shape)
print("Feature decision register shape:", feature_decision_df.shape)
print("Recommended features shape:", recommended_features_df.shape)
print("Feature audit shape:", feature_audit_df.shape)

display(df.head())
display(feature_decision_df.head())
display(recommended_features_df.head())

Selected raw COMPAS snapshot found at: /content/drive/MyDrive/Master_Research/07_Results/compas/01_manual_eda_compas/datasets/compas_selected_raw_dataset_snapshot.csv
Feature inclusion/exclusion register found at: /content/drive/MyDrive/Master_Research/07_Results/compas/01_manual_eda_compas/tables/feature_inclusion_exclusion_register.csv
Recommended modelling features found at: /content/drive/MyDrive/Master_Research/07_Results/compas/01_manual_eda_compas/tables/recommended_modelling_features.csv
Feature audit table found at: /content/drive/MyDrive/Master_Research/07_Results/compas/01_manual_eda_compas/tables/feature_audit_table.csv
Raw selected dataset shape: (18316, 40)
Feature decision register shape: (48, 5)
Recommended features shape: (10, 1)
Feature audit shape: (40, 21)


,id,name,first,last,sex,dob,age,age_cat,race,juv_fel_count,decile_score,juv_misd_count,juv_other_count,priors_count,days_b_screening_arrest,c_jail_in,c_jail_out,c_days_from_compas,c_charge_degree,c_charge_desc,is_recid,r_charge_degree,r_days_from_arrest,r_offense_date,r_charge_desc,r_jail_in,violent_recid,is_violent_recid,vr_charge_degree,vr_offense_date,vr_charge_desc,type_of_assessment,decile_score.1,score_text,screening_date,v_type_of_assessment,v_decile_score,v_score_text,priors_count.1,event
0,1.0000,miguel hernandez,miguel,hernandez,Male,18/04/1947,69,Greater than 45,Other,0,1,0,0,0,-1.0000,13/08/2013 6:03,14/08/2013 5:41,1.0000,(F3),Aggravated Assault w/Firearm,0,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,Risk of Recidivism,1,Low,14/08/2013,Risk of Violence,1,Low,0,0
1,2.0000,miguel hernandez,miguel,hernandez,Male,18/04/1947,69,Greater than 45,Other,0,1,0,0,0,-1.0000,13/08/2013 6:03,14/08/2013 5:41,1.0000,(F3),Aggravated Assault w/Firearm,0,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,Risk of Recidivism,1,Low,14/08/2013,Risk of Violence,1,Low,0,0
2,3.0000,michael ryan,michael,ryan,Male,06/02/1985,31,25 - 45,Caucasian,0,5,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,-1,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,Risk of Recidivism,5,Medium,31/12/2014,Risk of Violence,2,Low,0,0
3,4.0000,kevon dixon,kevon,dixon,Male,22/01/1982,34,25 - 45,African-American,0,3,0,0,0,-1.0000,26/01/2013 3:45,05/02/2013 5:36,1.0000,(F3),Felony Battery w/Prior Convict,1,(F3),NaN,05/07/2013,Felony Battery (Dom Strang),NaN,NaN,1,(F3),05/07/2013,Felony Battery (Dom Strang),Risk of Recidivism,3,Low,27/01/2013,Risk of Violence,1,Low,0,1
4,5.0000,ed philo,ed,philo,Male,14/05/1991,24,Less than 25,African-American,0,4,0,1,4,-1.0000,13/04/2013 4:58,14/04/2013 7:02,1.0000,(F3),Possession of Cocaine,1,(M1),0.0000,16/06/2013,Driving Under The Influence,16/06/2013,NaN,0,NaN,NaN,NaN,Risk of Recidivism,4,Low,14/04/2013,Risk of Violence,3,Low,4,0


,feature,feature_description,modelling_decision,decision_group,reasoning
0,sex,Sex of the individual.,Keep,sensitive_demographic,Sensitive demographic variable retained for fa...
1,age,Age in years at the time of COMPAS screening o...,Keep,sensitive_demographic,Age in years is retained because it is more in...
2,age_cat,"Age category derived from age, typically Less ...",Remove,redundant_derived_feature,Derived directly from age. Removed to reduce r...
3,race,Race or ethnicity category recorded in the COM...,Keep,sensitive_demographic,Sensitive demographic variable retained for fa...
4,juv_fel_count,Number of juvenile felony offences.,Keep,criminal_history,Historical juvenile felony count available bef...


,recommended_feature
0,sex
1,age
2,race
3,juv_fel_count
4,juv_misd_count


## 9. Target Cleaning

The target is `is_recid`.

Only binary target rows are retained:

```text
0 = no recidivism
1 = recidivism
```

Rows with `-1` or missing values are excluded.

In [5]:
# =========================================================
# 9. Target Cleaning
# =========================================================

TARGET = "is_recid"

if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' not found. Available columns: {df.columns.tolist()}")

target_before_df = (
    df[TARGET]
    .value_counts(dropna=False)
    .rename_axis("target_value")
    .reset_index(name="count")
)

target_before_df["pct"] = (
    target_before_df["count"] / len(df) * 100
).round(4)

df_model = df[df[TARGET].isin([0, 1])].copy()

target_after_df = (
    df_model[TARGET]
    .value_counts(dropna=False)
    .rename_axis("target_value")
    .reset_index(name="count")
)

target_after_df["pct"] = (
    target_after_df["count"] / len(df_model) * 100
).round(4)

target_cleaning_summary_df = pd.DataFrame([
    {
        "step": "before_target_cleaning",
        "n_rows": len(df),
        "valid_target_rows": int(df[TARGET].isin([0, 1]).sum()),
        "invalid_or_missing_target_rows": int((~df[TARGET].isin([0, 1])).sum())
    },
    {
        "step": "after_target_cleaning",
        "n_rows": len(df_model),
        "valid_target_rows": int(df_model[TARGET].isin([0, 1]).sum()),
        "invalid_or_missing_target_rows": 0
    }
])

display(target_before_df)
display(target_after_df)
display(target_cleaning_summary_df)

save_df(target_before_df, "target_distribution_before_cleaning.csv")
save_df(target_after_df, "target_distribution_after_cleaning.csv")
save_df(target_cleaning_summary_df, "target_cleaning_summary.csv")

,target_value,count,pct
0,0,9079,49.5687
1,1,8417,45.9544
2,-1,820,4.4770


,target_value,count,pct
0,0,9079,51.8919
1,1,8417,48.1081


,step,n_rows,valid_target_rows,invalid_or_missing_target_rows
0,before_target_cleaning,18316,17496,820
1,after_target_cleaning,17496,17496,0


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/tables/target_distribution_before_cleaning.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/tables/target_distribution_after_cleaning.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/tables/target_cleaning_summary.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/tables/target_cleaning_summary.csv')

## 10. Final Feature Selection

The final feature set is controlled and intentionally small.

This is done to keep the COMPAS benchmark comparable with German Credit and Pima Diabetes while avoiding leakage and unstable high-cardinality encoding.

In [6]:
# =========================================================
# 10. Final Feature Selection
# =========================================================

manual_final_features = [
    "sex",
    "age",
    "race",
    "juv_fel_count",
    "juv_misd_count",
    "juv_other_count",
    "priors_count",
    "days_b_screening_arrest",
    "c_days_from_compas",
    "c_charge_degree"
]

if not recommended_features_df.empty and "recommended_feature" in recommended_features_df.columns:
    final_features = recommended_features_df["recommended_feature"].dropna().astype(str).tolist()
else:
    final_features = manual_final_features

final_features_present = [col for col in final_features if col in df_model.columns]
final_features_missing = [col for col in final_features if col not in df_model.columns]

if final_features_missing:
    print("Warning: These expected final features were not found:", final_features_missing)

X = df_model[final_features_present].copy()
y = df_model[TARGET].astype(int).copy()

print("Final selected features:", final_features_present)
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())
display(y.head())

Final selected features: ['sex', 'age', 'race', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count', 'days_b_screening_arrest', 'c_days_from_compas', 'c_charge_degree']
Feature matrix shape: (17496, 10)
Target shape: (17496,)


,sex,age,race,juv_fel_count,juv_misd_count,juv_other_count,priors_count,days_b_screening_arrest,c_days_from_compas,c_charge_degree
0,Male,69,Other,0,0,0,0,-1.0000,1.0000,(F3)
1,Male,69,Other,0,0,0,0,-1.0000,1.0000,(F3)
3,Male,34,African-American,0,0,0,0,-1.0000,1.0000,(F3)
4,Male,24,African-American,0,0,1,4,-1.0000,1.0000,(F3)
5,Male,24,African-American,0,0,1,4,-1.0000,1.0000,(F3)


,is_recid
0,0
1,0
3,1
4,1
5,1


## 11. Feature Selection Register

This section documents exactly what was kept and what was excluded.

In [7]:
# =========================================================
# 11. Feature Selection Register
# =========================================================

all_raw_features = [col for col in df_model.columns if col != TARGET]

feature_selection_rows = []

for col in all_raw_features:
    kept = col in final_features_present

    if not feature_decision_df.empty and col in feature_decision_df["feature"].values:
        decision_row = feature_decision_df[feature_decision_df["feature"] == col].iloc[0]
        decision_group = decision_row.get("decision_group", "not_available")
        reasoning = decision_row.get("reasoning", "No reasoning found.")
        eda_decision = decision_row.get("modelling_decision", "not_available")
        description = decision_row.get("feature_description", "No description found.")
    elif not feature_audit_df.empty and col in feature_audit_df["feature"].values:
        audit_row = feature_audit_df[feature_audit_df["feature"] == col].iloc[0]
        decision_group = audit_row.get("decision_group", "not_available")
        reasoning = audit_row.get("reasoning", "No reasoning found.")
        eda_decision = audit_row.get("modelling_decision", "not_available")
        description = audit_row.get("feature_description", "No description found.")
    else:
        decision_group = "manual_final_feature" if kept else "not_selected"
        reasoning = "Kept as part of final agreed modelling feature set." if kept else "Excluded because it is not part of the final agreed modelling feature set."
        eda_decision = "Keep" if kept else "Remove"
        description = "No EDA description found."

    feature_selection_rows.append({
        "feature": col,
        "feature_description": description,
        "retained_for_modelling": kept,
        "eda_decision": eda_decision,
        "decision_group": decision_group,
        "reasoning": reasoning,
        "data_type": "numeric" if pd.api.types.is_numeric_dtype(df_model[col]) else "categorical",
        "missing_pct": round(float(df_model[col].isna().mean() * 100), 4),
        "n_unique": int(df_model[col].nunique(dropna=False))
    })

feature_selection_register_df = pd.DataFrame(feature_selection_rows)

display(feature_selection_register_df)

save_df(feature_selection_register_df, "feature_selection_register.csv")

,feature,feature_description,retained_for_modelling,eda_decision,decision_group,reasoning,data_type,missing_pct,n_unique
0,id,Internal row or person identifier.,False,Remove,identifier,Internal identifier with no modelling meaning.,numeric,39.7462,10543
1,name,Full name of the individual.,False,Remove,identifier,Personal identifier.,categorical,0.0000,10217
2,first,First name of the individual.,False,Remove,identifier,Personal identifier.,categorical,0.0000,3669
3,last,Last name of the individual.,False,Remove,identifier,Personal identifier.,categorical,0.0000,5331
4,sex,Sex of the individual.,True,Keep,sensitive_demographic,Sensitive demographic variable retained for fa...,categorical,0.0000,2
5,dob,Date of birth of the individual.,False,Remove,identifier_or_redundant,Date of birth is identifying and age is alread...,categorical,0.0000,7164
6,age,Age in years at the time of COMPAS screening o...,True,Keep,sensitive_demographic,Age in years is retained because it is more in...,numeric,0.0000,65
7,age_cat,"Age category derived from age, typically Less ...",False,Remove,redundant_derived_feature,Derived directly from age. Removed to reduce r...,categorical,0.0000,3
8,race,Race or ethnicity category recorded in the COM...,True,Keep,sensitive_demographic,Sensitive demographic variable retained for fa...,categorical,0.0000,6
9,juv_fel_count,Number of juvenile felony offences.,True,Keep,criminal_history,Historical juvenile felony count available bef...,numeric,0.0000,12


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/tables/feature_selection_register.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/tables/feature_selection_register.csv')

In [8]:
# =========================================================
# 12. Validate Explicit Exclusions
# =========================================================

should_be_excluded = [
    "age_cat",
    "c_charge_desc",
    "r_charge_degree",
    "r_charge_desc",
    "r_days_from_arrest",
    "violent_recid",
    "vr_charge_degree",
    "vr_charge_desc",
    "decile_score",
    "score_text",
    "v_decile_score",
    "v_score_text",
    "decile_score.1",
    "priors_count.1",
    "type_of_assessment",
    "v_type_of_assessment",
    "id",
    "name",
    "first",
    "last",
    "dob",
    "case_number",
    "c_case_number",
    "r_case_number",
    "vr_case_number"
]

explicit_exclusion_validation_df = pd.DataFrame([
    {
        "feature": col,
        "present_in_raw_data": col in df_model.columns,
        "present_in_final_X": col in X.columns,
        "exclusion_passed": col not in X.columns
    }
    for col in should_be_excluded
])

display(explicit_exclusion_validation_df)

save_df(explicit_exclusion_validation_df, "explicit_exclusion_validation.csv")

if not explicit_exclusion_validation_df["exclusion_passed"].all():
    raise ValueError("One or more explicitly excluded features are still present in X.")

,feature,present_in_raw_data,present_in_final_X,exclusion_passed
0,age_cat,True,False,True
1,c_charge_desc,True,False,True
2,r_charge_degree,True,False,True
3,r_charge_desc,True,False,True
4,r_days_from_arrest,True,False,True
5,violent_recid,True,False,True
6,vr_charge_degree,True,False,True
7,vr_charge_desc,True,False,True
8,decile_score,True,False,True
9,score_text,True,False,True


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/tables/explicit_exclusion_validation.csv


## 13. Train/Test Split

The split is stratified by the target to preserve the recidivism/no-recidivism class balance across train and test sets.

In [9]:
# =========================================================
# 13. Train/Test Split
# =========================================================

X_train_original, X_test_original, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

split_summary_df = pd.DataFrame([
    {
        "dataset": "train",
        "n_rows": len(X_train_original),
        "target_positive_count": int(y_train.sum()),
        "target_negative_count": int((y_train == 0).sum()),
        "target_positive_rate": round(float(y_train.mean()), 4)
    },
    {
        "dataset": "test",
        "n_rows": len(X_test_original),
        "target_positive_count": int(y_test.sum()),
        "target_negative_count": int((y_test == 0).sum()),
        "target_positive_rate": round(float(y_test.mean()), 4)
    }
])

display(split_summary_df)

save_df(split_summary_df, "train_test_split_summary.csv")

,dataset,n_rows,target_positive_count,target_negative_count,target_positive_rate
0,train,13996,6733,7263,0.4811
1,test,3500,1684,1816,0.4811


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/tables/train_test_split_summary.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/tables/train_test_split_summary.csv')

## 14. Build Preprocessing Pipeline

The preprocessing design follows the German Credit and Pima Diabetes approach:

- numeric features: median imputation + RobustScaler;
- categorical features: missing category imputation + One-Hot Encoding.

In [10]:
# =========================================================
# 14. Build Preprocessing Pipeline
# =========================================================

numeric_features = X_train_original.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train_original.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", get_onehot_encoder())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

preprocessing_design_df = pd.DataFrame([
    {
        "feature_type": "numeric",
        "n_features": len(numeric_features),
        "features": ", ".join(numeric_features),
        "imputation": "median",
        "scaling_or_encoding": "RobustScaler"
    },
    {
        "feature_type": "categorical",
        "n_features": len(categorical_features),
        "features": ", ".join(categorical_features),
        "imputation": "Missing category",
        "scaling_or_encoding": "OneHotEncoder(handle_unknown='ignore')"
    }
])

display(preprocessing_design_df)

save_df(preprocessing_design_df, "preprocessing_design.csv")

,feature_type,n_features,features,imputation,scaling_or_encoding
0,numeric,7,"age, juv_fel_count, juv_misd_count, juv_other_...",median,RobustScaler
1,categorical,3,"sex, race, c_charge_degree",Missing category,OneHotEncoder(handle_unknown='ignore')


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/tables/preprocessing_design.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/tables/preprocessing_design.csv')

In [11]:
# =========================================================
# 15. Fit Preprocessor and Transform Data
# =========================================================

X_train_array = preprocessor.fit_transform(X_train_original)
X_test_array = preprocessor.transform(X_test_original)

feature_names = get_feature_names_from_preprocessor(
    preprocessor,
    numeric_features,
    categorical_features
)

X_train_preprocessed = pd.DataFrame(
    X_train_array,
    columns=feature_names,
    index=X_train_original.index
)

X_test_preprocessed = pd.DataFrame(
    X_test_array,
    columns=feature_names,
    index=X_test_original.index
)

print("X_train_original shape:", X_train_original.shape)
print("X_test_original shape:", X_test_original.shape)
print("X_train_preprocessed shape:", X_train_preprocessed.shape)
print("X_test_preprocessed shape:", X_test_preprocessed.shape)

display(X_train_preprocessed.head())

X_train_original shape: (13996, 10)
X_test_original shape: (3500, 10)
X_train_preprocessed shape: (13996, 30)
X_test_preprocessed shape: (3500, 30)


,age,juv_fel_count,juv_misd_count,juv_other_count,priors_count,days_b_screening_arrest,c_days_from_compas,sex_Female,sex_Male,race_African-American,race_Asian,race_Caucasian,race_Hispanic,race_Native American,race_Other,c_charge_degree_(CO3),c_charge_degree_(CT),c_charge_degree_(F1),c_charge_degree_(F2),c_charge_degree_(F3),c_charge_degree_(F5),c_charge_degree_(F6),c_charge_degree_(F7),c_charge_degree_(M1),c_charge_degree_(M2),c_charge_degree_(MO3),c_charge_degree_(NI0),c_charge_degree_(TCX),c_charge_degree_(X),c_charge_degree_Missing
12010,0.3750,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1714,0.1875,0.0000,0.0000,1.0000,2.6000,-3.0000,4.0000,1.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
12344,-0.0625,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
10372,0.8750,0.0000,0.0000,0.0000,-0.4000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
10608,0.1250,0.0000,0.0000,0.0000,2.2000,1.0000,0.0000,0.0000,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


## 16. Validation Checks

This section validates that the preprocessing outputs are usable by downstream notebooks.

In [12]:
# =========================================================
# 16. Validate Preprocessing Outputs
# =========================================================

validation_rows = []

validation_rows.append({
    "check": "Train original and y_train aligned",
    "passed": len(X_train_original) == len(y_train)
})

validation_rows.append({
    "check": "Test original and y_test aligned",
    "passed": len(X_test_original) == len(y_test)
})

validation_rows.append({
    "check": "Train preprocessed and y_train aligned",
    "passed": len(X_train_preprocessed) == len(y_train)
})

validation_rows.append({
    "check": "Test preprocessed and y_test aligned",
    "passed": len(X_test_preprocessed) == len(y_test)
})

validation_rows.append({
    "check": "No missing values in X_train_preprocessed",
    "passed": X_train_preprocessed.isna().sum().sum() == 0
})

validation_rows.append({
    "check": "No missing values in X_test_preprocessed",
    "passed": X_test_preprocessed.isna().sum().sum() == 0
})

validation_rows.append({
    "check": "Train and test preprocessed columns match",
    "passed": list(X_train_preprocessed.columns) == list(X_test_preprocessed.columns)
})

validation_rows.append({
    "check": "Target is binary after cleaning",
    "passed": sorted(y.unique().tolist()) == [0, 1]
})

validation_rows.append({
    "check": "Only agreed final features retained",
    "passed": set(X.columns.tolist()) == set(final_features_present)
})

validation_rows.append({
    "check": "No COMPAS score outputs retained",
    "passed": not any(col in X.columns for col in [
        "decile_score", "score_text", "v_decile_score", "v_score_text", "decile_score.1"
    ])
})

validation_rows.append({
    "check": "No future-event leakage variables retained",
    "passed": not any(col in X.columns for col in [
        "r_charge_degree", "r_charge_desc", "r_days_from_arrest",
        "vr_charge_degree", "vr_charge_desc", "violent_recid"
    ])
})

validation_rows.append({
    "check": "No identifier variables retained",
    "passed": not any(col in X.columns for col in [
        "id", "name", "first", "last", "dob", "case_number", "c_case_number",
        "r_case_number", "vr_case_number"
    ])
})

preprocessing_validation_df = pd.DataFrame(validation_rows)

display(preprocessing_validation_df)

save_df(preprocessing_validation_df, "preprocessing_validation_checks.csv")

if not preprocessing_validation_df["passed"].all():
    raise ValueError("One or more preprocessing validation checks failed.")

,check,passed
0,Train original and y_train aligned,True
1,Test original and y_test aligned,True
2,Train preprocessed and y_train aligned,True
3,Test preprocessed and y_test aligned,True
4,No missing values in X_train_preprocessed,True
5,No missing values in X_test_preprocessed,True
6,Train and test preprocessed columns match,True
7,Target is binary after cleaning,True
8,Only agreed final features retained,True
9,No COMPAS score outputs retained,True


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/tables/preprocessing_validation_checks.csv


## 17. Save Standard Modelling Outputs

The saved file names intentionally match the German Credit and Pima Diabetes conventions.

In [13]:
# =========================================================
# 17. Save Standard Modelling Outputs
# =========================================================

# Preserve original row index for traceability only.
# This is not used as a modelling feature and is dropped downstream where necessary.
X_train_original_out = X_train_original.copy()
X_test_original_out = X_test_original.copy()

X_train_original_out.insert(0, "original_index", X_train_original_out.index)
X_test_original_out.insert(0, "original_index", X_test_original_out.index)

X_train_original_out.to_csv(DATASETS_PATH / "X_train_original.csv", index=False)
X_test_original_out.to_csv(DATASETS_PATH / "X_test_original.csv", index=False)

X_train_preprocessed.to_csv(DATASETS_PATH / "X_train_preprocessed.csv", index=False)
X_test_preprocessed.to_csv(DATASETS_PATH / "X_test_preprocessed.csv", index=False)

pd.DataFrame({"target": y_train.reset_index(drop=True)}).to_csv(DATASETS_PATH / "y_train.csv", index=False)
pd.DataFrame({"target": y_test.reset_index(drop=True)}).to_csv(DATASETS_PATH / "y_test.csv", index=False)

# Compatibility samples in tables folder for quick review
X_train_original_out.head(20).to_csv(TABLES_PATH / "X_train_original_sample.csv", index=False)
X_test_original_out.head(20).to_csv(TABLES_PATH / "X_test_original_sample.csv", index=False)
X_train_preprocessed.head(20).to_csv(TABLES_PATH / "X_train_preprocessed_sample.csv", index=False)
X_test_preprocessed.head(20).to_csv(TABLES_PATH / "X_test_preprocessed_sample.csv", index=False)

print("Standard dataset outputs saved.")

Standard dataset outputs saved.


In [14]:
# =========================================================
# 18. Save Preprocessor and Feature Metadata
# =========================================================

preprocessor_path = MODELS_PATH / "preprocessor.pkl"
joblib.dump(preprocessor, preprocessor_path)

# Compatibility alias in case older scripts search for this name
joblib.dump(preprocessor, MODELS_PATH / "preprocessing_pipeline.joblib")

preprocessed_feature_registry_df = pd.DataFrame({
    "preprocessed_feature": feature_names
})

preprocessed_feature_registry_df["source_transformer"] = np.where(
    preprocessed_feature_registry_df["preprocessed_feature"].isin(numeric_features),
    "numeric",
    "categorical_onehot"
)

preprocessing_feature_registry_df = feature_selection_register_df.copy()

preprocessing_feature_registry_df["included_in_original_feature_space"] = (
    preprocessing_feature_registry_df["feature"].isin(X.columns)
)

preprocessing_feature_registry_df["numeric_feature"] = (
    preprocessing_feature_registry_df["feature"].isin(numeric_features)
)

preprocessing_feature_registry_df["categorical_feature"] = (
    preprocessing_feature_registry_df["feature"].isin(categorical_features)
)

display(preprocessed_feature_registry_df.head())
display(preprocessing_feature_registry_df.head())

save_df(preprocessed_feature_registry_df, "preprocessed_feature_registry.csv")
save_df(preprocessing_feature_registry_df, "preprocessing_feature_registry.csv")

print("Saved preprocessor:", preprocessor_path)

,preprocessed_feature,source_transformer
0,age,numeric
1,juv_fel_count,numeric
2,juv_misd_count,numeric
3,juv_other_count,numeric
4,priors_count,numeric


,feature,feature_description,retained_for_modelling,eda_decision,decision_group,reasoning,data_type,missing_pct,n_unique,included_in_original_feature_space,numeric_feature,categorical_feature
0,id,Internal row or person identifier.,False,Remove,identifier,Internal identifier with no modelling meaning.,numeric,39.7462,10543,False,False,False
1,name,Full name of the individual.,False,Remove,identifier,Personal identifier.,categorical,0.0000,10217,False,False,False
2,first,First name of the individual.,False,Remove,identifier,Personal identifier.,categorical,0.0000,3669,False,False,False
3,last,Last name of the individual.,False,Remove,identifier,Personal identifier.,categorical,0.0000,5331,False,False,False
4,sex,Sex of the individual.,True,Keep,sensitive_demographic,Sensitive demographic variable retained for fa...,categorical,0.0000,2,True,False,True


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/tables/preprocessed_feature_registry.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/tables/preprocessing_feature_registry.csv
Saved preprocessor: /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/models/preprocessor.pkl


## 19. Save Preprocessing Summary

In [15]:
# =========================================================
# 19. Save Preprocessing Summary
# =========================================================

preprocessing_summary_df = pd.DataFrame([
    {
        "dataset_name": DATASET_NAME,
        "target": TARGET,
        "target_positive_class": 1,
        "target_positive_interpretation": "Recidivism",
        "target_negative_class": 0,
        "target_negative_interpretation": "No recidivism",
        "n_rows_before_target_cleaning": len(df),
        "n_rows_after_target_cleaning": len(df_model),
        "n_original_features_retained": X.shape[1],
        "n_preprocessed_features": X_train_preprocessed.shape[1],
        "n_numeric_features": len(numeric_features),
        "n_categorical_features": len(categorical_features),
        "numeric_features": ", ".join(numeric_features),
        "categorical_features": ", ".join(categorical_features),
        "final_features": ", ".join(final_features_present),
        "train_rows": len(X_train_original),
        "test_rows": len(X_test_original),
        "test_size": 0.20,
        "random_state": RANDOM_STATE,
        "numeric_imputation": "median",
        "numeric_scaling": "RobustScaler",
        "categorical_imputation": "Missing category",
        "categorical_encoding": "OneHotEncoder(handle_unknown='ignore')",
        "compas_score_outputs_excluded": True,
        "future_event_leakage_excluded": True,
        "identifiers_excluded": True,
        "sensitive_features_retained_for_audit": True,
        "age_cat_removed_due_to_redundancy": "age_cat" not in X.columns,
        "c_charge_desc_removed_due_to_high_cardinality": "c_charge_desc" not in X.columns
    }
])

display(preprocessing_summary_df)

save_df(preprocessing_summary_df, "preprocessing_summary.csv")

,dataset_name,target,target_positive_class,target_positive_interpretation,target_negative_class,target_negative_interpretation,n_rows_before_target_cleaning,n_rows_after_target_cleaning,n_original_features_retained,n_preprocessed_features,n_numeric_features,n_categorical_features,numeric_features,categorical_features,final_features,train_rows,test_rows,test_size,random_state,numeric_imputation,numeric_scaling,categorical_imputation,categorical_encoding,compas_score_outputs_excluded,future_event_leakage_excluded,identifiers_excluded,sensitive_features_retained_for_audit,age_cat_removed_due_to_redundancy,c_charge_desc_removed_due_to_high_cardinality
0,compas,is_recid,1,Recidivism,0,No recidivism,18316,17496,10,30,7,3,"age, juv_fel_count, juv_misd_count, juv_other_...","sex, race, c_charge_degree","sex, age, race, juv_fel_count, juv_misd_count,...",13996,3500,0.2000,42,median,RobustScaler,Missing category,OneHotEncoder(handle_unknown='ignore'),True,True,True,True,True,True


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/tables/preprocessing_summary.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/tables/preprocessing_summary.csv')

## 20. Save Consolidated Preprocessing Workbook and JSON Assumptions

In [16]:
# =========================================================
# 20. Save Consolidated Preprocessing Workbook and JSON
# =========================================================

save_excel(
    {
        "summary": preprocessing_summary_df,
        "target_before": target_before_df,
        "target_after": target_after_df,
        "target_cleaning": target_cleaning_summary_df,
        "split_summary": split_summary_df,
        "feature_selection": feature_selection_register_df,
        "explicit_exclusions": explicit_exclusion_validation_df,
        "preprocessing_design": preprocessing_design_df,
        "feature_registry": preprocessing_feature_registry_df,
        "preprocessed_features": preprocessed_feature_registry_df,
        "validation": preprocessing_validation_df
    },
    "data_preprocessing_compas_outputs.xlsx"
)

assumptions = {
    "dataset_name": DATASET_NAME,
    "target": TARGET,
    "target_mapping": {
        "0": "No recidivism",
        "1": "Recidivism"
    },
    "invalid_targets_removed": True,
    "final_features": final_features_present,
    "excluded_feature_logic": {
        "identifiers": "excluded",
        "compas_score_outputs": "excluded to avoid circularity",
        "future_event_variables": "excluded to avoid target leakage",
        "duplicate_variables": "excluded",
        "constant_or_noninformative_variables": "excluded",
        "high_cardinality_charge_description": "excluded",
        "age_category": "excluded because age is retained"
    },
    "sensitive_features_retained": [
        col for col in ["sex", "age", "race"]
        if col in final_features_present
    ],
    "numeric_preprocessing": "median imputation + RobustScaler",
    "categorical_preprocessing": "Missing category imputation + OneHotEncoder",
    "train_test_split": "80/20 stratified split",
    "random_state": RANDOM_STATE,
    "research_reasoning": (
        "The COMPAS preprocessing strategy creates an independent recidivism prediction dataset "
        "using assessment-time information only. COMPAS score outputs and future-event variables "
        "are excluded. Sensitive variables are retained to support fairness and proxy-risk analysis."
    )
}

with open(LOGS_PATH / "preprocessing_assumptions.json", "w", encoding="utf-8") as f:
    json.dump(assumptions, f, indent=4)

with open(TABLES_PATH / "preprocessing_assumptions.json", "w", encoding="utf-8") as f:
    json.dump(assumptions, f, indent=4)

print("Preprocessing workbook and assumptions JSON saved.")

Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/tables/data_preprocessing_compas_outputs.xlsx
Preprocessing workbook and assumptions JSON saved.


## 21. Interpretation Notes

### Simple Interpretation

This notebook cleaned the COMPAS data into a modelling-ready format.

It kept only the agreed final feature set and removed variables that could cause leakage, circularity, redundancy or unstable explanations.

### Technical Interpretation

The notebook created both original and preprocessed feature spaces:

- original features are used for full pipelines and DiCE counterfactual recourse;
- preprocessed features are used for estimator-only models, SHAP and LIME.

### Research Contribution

This design keeps the COMPAS preprocessing pipeline consistent with German Credit and Pima Diabetes while documenting COMPAS-specific fairness, leakage and feature-selection assumptions.

In [17]:
# =========================================================
# 21. Final Completion Message
# =========================================================

print("=" * 100)
print("02 Data Preprocessing – COMPAS completed successfully.")
print("=" * 100)

print(f"Datasets saved to: {DATASETS_PATH}")
print(f"Tables saved to:   {TABLES_PATH}")
print(f"Models saved to:   {MODELS_PATH}")

print("\nKey outputs:")
print("- X_train_original.csv")
print("- X_test_original.csv")
print("- X_train_preprocessed.csv")
print("- X_test_preprocessed.csv")
print("- y_train.csv")
print("- y_test.csv")
print("- preprocessor.pkl")
print("- preprocessing_pipeline.joblib")
print("- preprocessing_summary.csv")
print("- feature_selection_register.csv")
print("- explicit_exclusion_validation.csv")
print("- preprocessing_feature_registry.csv")
print("- data_preprocessing_compas_outputs.xlsx")

02 Data Preprocessing – COMPAS completed successfully.
Datasets saved to: /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/datasets
Tables saved to:   /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/tables
Models saved to:   /content/drive/MyDrive/Master_Research/07_Results/compas/02_data_preprocessing_compas/models

Key outputs:
- X_train_original.csv
- X_test_original.csv
- X_train_preprocessed.csv
- X_test_preprocessed.csv
- y_train.csv
- y_test.csv
- preprocessor.pkl
- preprocessing_pipeline.joblib
- preprocessing_summary.csv
- feature_selection_register.csv
- explicit_exclusion_validation.csv
- preprocessing_feature_registry.csv
- data_preprocessing_compas_outputs.xlsx
